# TrendSummary v1.2 — Yahoo Consensus Trend Analysis

## Purpose
Analyzes **revision trends** in Yahoo Consensus data across multiple time windows (1W, 2W, 1M, 2M, 3M) to identify stocks where analyst estimates are changing.

## Input
- All `*_YahooConsensus.csv` files in the folder
- Filters to tickers with >= 5 analysts (`MIN_ANALYSTS`)

## Output File
`YYYYMMDD_YahooConsensusTrendAnalysis.xlsx` with **10 sheets**

---

## Sheet Descriptions

### **Base Sheets (TOP 10 per category)**

**1. TrendAnalysis**  
TP(Min/Avg/Max) revisions across 5 windows × 3 metrics × 2 directions = 30 categories  
Shows TOP 10 upward and TOP 10 downward movers per category  
Columns: Header, Direction, Rank, YAHOO Ticker, FIN Yr, Change, Price_Change, #Count

**2. RevenueTrendAnalysis**  
Rev.(Min/Avg/Max) revisions for Y1 & Y2 across 5 windows  
Shows TOP 10 upward/downward per category

**3. EPSTrendAnalysis**  
EPS(Min/Avg/Max) revisions for Y1 & Y2 across 5 windows  
Shows TOP 10 upward/downward per category

---

### **Insight Sheets (ALL tickers)**

**4. Summary**  
Combines all base sheets into one master list  
Contains ALL ranked results (not just top 10)  
Use this as the source for further analysis

**5. InsightsFrequency**  
How many times each ticker appears across ALL categories  
- `Frequency` — total appearances  
- `Upward_Count` — appearances in upward revision lists  
- `Downward_Count` — appearances in downward revision lists  
- `Net_Direction` — Upward - Downward (positive = net bullish)  

**Sorted by:** Frequency DESC  
**Use case:** Find tickers with broad-based consensus changes

**6. InsightsRankStrength**  
Sum of `1/Rank` scores across all appearances (same method as your other notebooks)  
- `Total_Score` — sum of all 1/Rank values  
- `Avg_Score` — Total_Score / Appearance_Count  
- `Upward_Score` — sum of 1/Rank for upward appearances only  
- `Downward_Score` — sum of 1/Rank for downward appearances only  
- `Net_Score` — Upward - Downward  

**Sorted by:** Total_Score DESC  
**Use case:** Identify highest-conviction revision stories

**7. InsightsMomentumType**  
Categorizes tickers by revision pattern across time windows  
- `Accelerating` — revisions getting stronger (recent windows > older windows)  
- `Decelerating` — revisions weakening  
- `Sustained` — similar magnitude across all windows (low variance)  
- `Recent` — only appears in 1W/2W windows  
- `Downward_Only` — only appears in downward lists  
- `Single_Window` — appears in only one time window  

**Sorted by:** Momentum_Type, then Avg_Revision DESC  
**Use case:** Understand *when* the consensus shift happened

**8. InsightsPriceVsEstimate**  
Finds **asymmetric opportunities** — estimate and price moving in opposite directions  
- `EstimateUp_PriceDown` — analysts upgrading but stock falling (potential buy)  
- `EstimateDown_PriceUp` — analysts downgrading but stock rising (potential short)  
- `Divergence` — abs(Change - Price_Change) — size of the gap  

**Sorted by:** Divergence DESC  
**Use case:** Identify mispricings where market hasn't reacted to estimate changes yet

**9. InsightsCategorySpread**  
How many category *types* (TP / Revenue / EPS) does each ticker appear in  
- `TP_Count` — appearances in TP categories  
- `Rev_Count` — appearances in Revenue categories  
- `EPS_Count` — appearances in EPS categories  
- `Total_Categories` — how many of the 3 types (0-3)  

**Sorted by:** Total_Categories DESC  
**Use case:** Tickers in all 3 categories = comprehensive consensus shift (stronger signal)

**10. InsightsYearWise**  
Compares near-term (Y1) vs long-term (Y2) revision strength  
- `Y1_Score` — sum of 1/Rank for FY1 appearances  
- `Y2_Score` — sum of 1/Rank for FY2 appearances  
- `Preferred_Year` — Y1 / Y2 / Both (based on which score is 20%+ higher)  

**Sorted by:** Y1_Score DESC  
**Use case:** See if analysts more bullish on near-term or long-term prospects

---

## Key Parameters
- `MIN_ANALYSTS = 5` — minimum analyst count to include ticker  
- `TOP_N = 10` — number of tickers shown per category in base sheets  
- `PERIODS = ["1W", "2W", "1M", "2M", "3M"]` — lookback windows

## Methodology
- **Revision calculation:** Latest value / Past value - 1  
- **Price change:** Measured over same window as estimate revision  
- **Rank scoring:** 1/Rank composite (Rank 1 gets score 1.0, Rank 10 gets 0.1)  
- **Direction:** Sorted ascending for downward (worst performers first), descending for upward (best first)

In [5]:
import pandas as pd
import glob
import numpy as np
from datetime import timedelta
from dateutil.relativedelta import relativedelta

# ================= CONFIG =================
FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"
MIN_ANALYSTS = 5
TOP_N = 10

PERIODS = ["1W", "2W", "1M", "2M", "3M"]

# ================= LOAD DATA =================
files = sorted(glob.glob(f"{FOLDER}\\*_YahooConsensus.csv"))

df_all = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_all["Date"] = pd.to_datetime(df_all["Date"])
df_all = df_all[df_all["#Count"] >= MIN_ANALYSTS].copy()

latest_date = df_all["Date"].max()
latest_date_str = latest_date.strftime("%Y%m%d")

output_file = f"{FOLDER}\\{latest_date_str}_YahooConsensusTrendAnalysis.xlsx"

# ================= DATE HELPERS =================
def get_period_target(latest, label):
    num = int(label[:-1])
    unit = label[-1]

    if unit == "W":
        return latest - timedelta(weeks=num)
    elif unit == "M":
        return latest - relativedelta(months=num)
    else:
        raise ValueError(f"Invalid period label: {label}")

def get_asof_date(dates, target_date):
    eligible = dates[dates <= target_date]
    return eligible.max() if not eligible.empty else None

# ================= TARGET PRICE REVISION =================
def compute_tp_revision(df, col, period_label):
    latest_date = df["Date"].max()
    target_date = get_period_target(latest_date, period_label)
    past_date = get_asof_date(df["Date"], target_date)

    if past_date is None:
        return pd.DataFrame()

    df_latest = df[df["Date"] == latest_date].drop_duplicates("YAHOO Ticker")
    df_past = df[df["Date"] == past_date].drop_duplicates("YAHOO Ticker")

    merged = df_latest.merge(df_past, on="YAHOO Ticker", suffixes=("_L", "_P"))

    merged = merged[(merged[f"{col}_P"] > 0) & (merged[f"{col}_L"] > 0)]
    if merged.empty:
        return pd.DataFrame()

    # TP Change
    merged["Change"] = merged[f"{col}_L"] / merged[f"{col}_P"] - 1

    # Price Change
    merged["Price_Change"] = merged["Current Price_L"] / merged["Current Price_P"] - 1

    # Clean inf/nan
    merged["Change"] = merged["Change"].replace([np.inf, -np.inf], 0).fillna(0)
    merged["Price_Change"] = merged["Price_Change"].replace([np.inf, -np.inf], 0).fillna(0)

    # Round to 4 decimals
    merged["Change"] = merged["Change"].round(4)
    merged["Price_Change"] = merged["Price_Change"].round(4)

    merged["#Count"] = merged["#Count_L"]
    merged["FIN Yr"] = ""  # TP has no FIN Yr

    return merged[["YAHOO Ticker", "FIN Yr", "Change", "Price_Change", "#Count"]]

# ================= REV/EPS REVISION =================
def compute_revision(df, col, period_label, fin_year):
    latest_date = df["Date"].max()
    target_date = get_period_target(latest_date, period_label)
    past_date = get_asof_date(df["Date"], target_date)

    if past_date is None:
        return pd.DataFrame()

    df_latest = df[(df["Date"] == latest_date) & (df["FIN Yr"] == fin_year)]
    df_past = df[(df["Date"] == past_date) & (df["FIN Yr"] == fin_year)]

    if df_latest.empty or df_past.empty:
        return pd.DataFrame()

    df_latest = df_latest.drop_duplicates(["YAHOO Ticker", "FIN Yr"])
    df_past = df_past.drop_duplicates(["YAHOO Ticker", "FIN Yr"])

    merged = df_latest.merge(df_past, on=["YAHOO Ticker", "FIN Yr"], suffixes=("_L", "_P"))

    merged = merged[(merged[f"{col}_P"] > 0) & (merged[f"{col}_L"] > 0)]
    if merged.empty:
        return pd.DataFrame()

    # Rev/EPS Change
    merged["Change"] = merged[f"{col}_L"] / merged[f"{col}_P"] - 1

    # Price Change
    merged["Price_Change"] = merged["Current Price_L"] / merged["Current Price_P"] - 1

    # Clean inf/nan
    merged["Change"] = merged["Change"].replace([np.inf, -np.inf], 0).fillna(0)
    merged["Price_Change"] = merged["Price_Change"].replace([np.inf, -np.inf], 0).fillna(0)

    # Round to 4 decimals
    merged["Change"] = merged["Change"].round(4)
    merged["Price_Change"] = merged["Price_Change"].round(4)

    merged["#Count"] = merged["#Count_L"]

    return merged[["YAHOO Ticker", "FIN Yr", "Change", "Price_Change", "#Count"]]

# ================= IDENTIFY Y1 & Y2 =================
fy_sorted = sorted(df_all["FIN Yr"].unique())
Y1 = fy_sorted[0]
Y2 = fy_sorted[1]

YMAP = {Y1: "Y1", Y2: "Y2"}

# ================= BUILD SHEETS =================
tp_master = []
rev_master = []
eps_master = []

# ---------- TARGET PRICE ----------
for period in PERIODS:
    for col, label in [
        ("TP(Min)", "TP Min Revision"),
        ("TP(Avg)", "TP Avg Revision"),
        ("TP(Max)", "TP Max Revision"),
    ]:

        header = f"{period} {label}"
        df_out = compute_tp_revision(df_all, col, period)

        if df_out.empty:
            continue

        # Upward
        df_up = df_out.sort_values("Change", ascending=False).head(TOP_N).reset_index(drop=True)
        df_up.insert(0, "Rank", df_up.index + 1)
        df_up.insert(0, "Direction", "Upward")
        df_up.insert(0, "Header", header)

        # Downward
        df_down = df_out.sort_values("Change", ascending=True).head(TOP_N).reset_index(drop=True)
        df_down.insert(0, "Rank", df_down.index + 1)
        df_down.insert(0, "Direction", "Downward")
        df_down.insert(0, "Header", header)

        tp_master.append(df_up)
        tp_master.append(df_down)

# ---------- REVENUE ----------
for period in PERIODS:
    for fin_year in [Y1, Y2]:
        y_label = YMAP[fin_year]

        for col, label in [
            ("Rev.(Min)", "Rev. Min Revision"),
            ("Rev.(Avg)", "Rev. Avg Revision"),
            ("Rev.(Max)", "Rev. Max Revision"),
        ]:

            header = f"{period} {y_label} {label}"
            df_out = compute_revision(df_all, col, period, fin_year)

            if df_out.empty:
                continue

            df_up = df_out.sort_values("Change", ascending=False).head(TOP_N).reset_index(drop=True)
            df_up.insert(0, "Rank", df_up.index + 1)
            df_up.insert(0, "Direction", "Upward")
            df_up.insert(0, "Header", header)

            df_down = df_out.sort_values("Change", ascending=True).head(TOP_N).reset_index(drop=True)
            df_down.insert(0, "Rank", df_down.index + 1)
            df_down.insert(0, "Direction", "Downward")
            df_down.insert(0, "Header", header)

            rev_master.append(df_up)
            rev_master.append(df_down)

# ---------- EPS ----------
for period in PERIODS:
    for fin_year in [Y1, Y2]:
        y_label = YMAP[fin_year]

        for col, label in [
            ("EPS(Min)", "EPS Min Revision"),
            ("EPS(Avg)", "EPS Avg Revision"),
            ("EPS(Max)", "EPS Max Revision"),
        ]:

            header = f"{period} {y_label} {label}"
            df_out = compute_revision(df_all, col, period, fin_year)

            if df_out.empty:
                continue

            df_up = df_out.sort_values("Change", ascending=False).head(TOP_N).reset_index(drop=True)
            df_up.insert(0, "Rank", df_up.index + 1)
            df_up.insert(0, "Direction", "Upward")
            df_up.insert(0, "Header", header)

            df_down = df_out.sort_values("Change", ascending=True).head(TOP_N).reset_index(drop=True)
            df_down.insert(0, "Rank", df_down.index + 1)
            df_down.insert(0, "Direction", "Downward")
            df_down.insert(0, "Header", header)

            eps_master.append(df_up)
            eps_master.append(df_down)

# ================= CREATE SUMMARY SHEET =================
summary_df = pd.concat(
    [pd.concat(tp_master, ignore_index=True),
     pd.concat(rev_master, ignore_index=True),
     pd.concat(eps_master, ignore_index=True)],
    ignore_index=True
)

# ================= INSIGHTS 1: FREQUENCY =================
freq_df = (
    summary_df
    .groupby('YAHOO Ticker')
    .agg(
        Frequency=('Header', 'count'),
        Upward_Count=('Direction', lambda x: (x == 'Upward').sum()),
        Downward_Count=('Direction', lambda x: (x == 'Downward').sum())
    )
    .reset_index()
)
freq_df['Net_Direction'] = freq_df['Upward_Count'] - freq_df['Downward_Count']
freq_df = freq_df.sort_values('Frequency', ascending=False).reset_index(drop=True)

# ================= INSIGHTS 2: RANK STRENGTH =================
rank_strength_df = (
    summary_df
    .assign(Score=lambda x: 1 / x['Rank'])
    .groupby('YAHOO Ticker')
    .agg(
        Total_Score=('Score', 'sum'),
        Appearance_Count=('Score', 'count'),
        Upward_Score=('Score', lambda x: x[summary_df.loc[x.index, 'Direction'] == 'Upward'].sum()),
        Downward_Score=('Score', lambda x: x[summary_df.loc[x.index, 'Direction'] == 'Downward'].sum())
    )
    .reset_index()
)
rank_strength_df['Avg_Score'] = rank_strength_df['Total_Score'] / rank_strength_df['Appearance_Count']
rank_strength_df['Net_Score'] = rank_strength_df['Upward_Score'] - rank_strength_df['Downward_Score']
rank_strength_df = rank_strength_df.sort_values('Total_Score', ascending=False).reset_index(drop=True)

# ================= INSIGHTS 3: MOMENTUM TYPE =================
def categorize_momentum(ticker_df):
    """Categorize ticker by revision pattern across windows"""
    upward_df = ticker_df[ticker_df['Direction'] == 'Upward'].copy()
    if upward_df.empty:
        return pd.Series({
            'Momentum_Type': 'Downward_Only',
            'Windows_Appeared': '',
            'Avg_Revision': np.nan
        })
    
    # Extract window from Header (e.g., "1W Y1 Rev. Avg Revision" -> "1W")
    upward_df['Window'] = upward_df['Header'].str.extract(r'^(\d+[WM])')[0]
    windows = upward_df['Window'].unique()
    changes = upward_df['Change'].values
    
    # Categorize
    if len(windows) <= 2 and all(w in ['1W', '2W'] for w in windows):
        momentum_type = 'Recent'
    elif len(changes) >= 2:
        cv = np.std(changes) / np.mean(changes) if np.mean(changes) != 0 else 0
        if cv < 0.3:
            momentum_type = 'Sustained'
        else:
            # Check if accelerating (recent stronger)
            window_order = {'1W': 1, '2W': 2, '1M': 3, '2M': 4, '3M': 5}
            upward_df['window_num'] = upward_df['Window'].map(window_order)
            upward_df = upward_df.sort_values('window_num')
            if upward_df['Change'].iloc[0] > upward_df['Change'].iloc[-1]:
                momentum_type = 'Accelerating'
            else:
                momentum_type = 'Decelerating'
    else:
        momentum_type = 'Single_Window'
    
    return pd.Series({
        'Momentum_Type': momentum_type,
        'Windows_Appeared': ', '.join(sorted(windows)),
        'Avg_Revision': round(upward_df['Change'].mean(), 4)
    })

momentum_type_df = (
    summary_df
    .groupby('YAHOO Ticker')
    .apply(categorize_momentum)
    .reset_index()
)
momentum_type_df = momentum_type_df.sort_values(
    ['Momentum_Type', 'Avg_Revision'],
    ascending=[True, False]
).reset_index(drop=True)

# ================= INSIGHTS 4: PRICE VS ESTIMATE =================
price_vs_estimate_df = summary_df[
    ((summary_df['Change'] > 0) & (summary_df['Price_Change'] < 0)) |
    ((summary_df['Change'] < 0) & (summary_df['Price_Change'] > 0))
].copy()

price_vs_estimate_df['Divergence'] = (
    (price_vs_estimate_df['Change'] - price_vs_estimate_df['Price_Change']).abs()
)
price_vs_estimate_df['Opportunity_Type'] = price_vs_estimate_df.apply(
    lambda x: 'EstimateUp_PriceDown' if x['Change'] > 0 else 'EstimateDown_PriceUp',
    axis=1
)
price_vs_estimate_df = price_vs_estimate_df[
    ['YAHOO Ticker', 'Header', 'Direction', 'Change', 'Price_Change', 'Divergence', 'Opportunity_Type']
].sort_values('Divergence', ascending=False).reset_index(drop=True)

# ================= INSIGHTS 5: CATEGORY SPREAD =================
def categorize_header(header):
    """Determine if TP / Rev / EPS category"""
    if 'TP' in header:
        return 'TP'
    elif 'Rev' in header:
        return 'Rev'
    elif 'EPS' in header:
        return 'EPS'
    return 'Other'

summary_df['Category'] = summary_df['Header'].apply(categorize_header)

category_spread_df = (
    summary_df
    .groupby('YAHOO Ticker')['Category']
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
category_spread_df.columns.name = None
if 'TP' in category_spread_df.columns:
    category_spread_df.rename(columns={'TP': 'TP_Count'}, inplace=True)
else:
    category_spread_df['TP_Count'] = 0
if 'Rev' in category_spread_df.columns:
    category_spread_df.rename(columns={'Rev': 'Rev_Count'}, inplace=True)
else:
    category_spread_df['Rev_Count'] = 0
if 'EPS' in category_spread_df.columns:
    category_spread_df.rename(columns={'EPS': 'EPS_Count'}, inplace=True)
else:
    category_spread_df['EPS_Count'] = 0

category_spread_df['Total_Categories'] = (
    (category_spread_df['TP_Count'] > 0).astype(int) +
    (category_spread_df['Rev_Count'] > 0).astype(int) +
    (category_spread_df['EPS_Count'] > 0).astype(int)
)
category_spread_df = category_spread_df[
    ['YAHOO Ticker', 'TP_Count', 'Rev_Count', 'EPS_Count', 'Total_Categories']
].sort_values('Total_Categories', ascending=False).reset_index(drop=True)

# ================= INSIGHTS 6: YEAR-WISE =================
# Filter summary to only Rev/EPS (TP has no FIN Yr)
yearwise_df = summary_df[summary_df['FIN Yr'].isin([Y1, Y2])].copy()
yearwise_df['Score'] = 1 / yearwise_df['Rank']

y1_scores = (
    yearwise_df[yearwise_df['FIN Yr'] == Y1]
    .groupby('YAHOO Ticker')['Score']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'Y1_Score', 'count': 'Y1_Count'})
)

y2_scores = (
    yearwise_df[yearwise_df['FIN Yr'] == Y2]
    .groupby('YAHOO Ticker')['Score']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'Y2_Score', 'count': 'Y2_Count'})
)

yearwise_agg_df = y1_scores.join(y2_scores, how='outer').fillna(0).reset_index()
yearwise_agg_df['Preferred_Year'] = yearwise_agg_df.apply(
    lambda x: 'Y1' if x['Y1_Score'] > x['Y2_Score'] * 1.2
              else 'Y2' if x['Y2_Score'] > x['Y1_Score'] * 1.2
              else 'Both',
    axis=1
)
yearwise_agg_df = yearwise_agg_df.sort_values(
    'Y1_Score', ascending=False
).reset_index(drop=True)

# ================= WRITE TO EXCEL =================
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    # Original 3 sheets (TOP_N per category)
    pd.concat(tp_master, ignore_index=True).to_excel(
        writer, sheet_name="TrendAnalysis", index=False
    )
    pd.concat(rev_master, ignore_index=True).to_excel(
        writer, sheet_name="RevenueTrendAnalysis", index=False
    )
    pd.concat(eps_master, ignore_index=True).to_excel(
        writer, sheet_name="EPSTrendAnalysis", index=False
    )
    
    # NEW: Summary + 6 Insights (ALL tickers)
    summary_df[['Header', 'Direction', 'Rank', 'YAHOO Ticker', 'FIN Yr', 'Change', 'Price_Change', '#Count']].to_excel(
        writer, sheet_name="Summary", index=False
    )
    freq_df.to_excel(writer, sheet_name="InsightsFrequency", index=False)
    rank_strength_df.to_excel(writer, sheet_name="InsightsRankStrength", index=False)
    momentum_type_df.to_excel(writer, sheet_name="InsightsMomentumType", index=False)
    price_vs_estimate_df.to_excel(writer, sheet_name="InsightsPriceVsEstimate", index=False)
    category_spread_df.to_excel(writer, sheet_name="InsightsCategorySpread", index=False)
    yearwise_agg_df.to_excel(writer, sheet_name="InsightsYearWise", index=False)

print(f"\n✅ Excel file created with 10 sheets:")
print(f"   {output_file}")
print(f"\nSheets:")
print(f"   1. TrendAnalysis (TOP {TOP_N} per window)")
print(f"   2. RevenueTrendAnalysis (TOP {TOP_N} per window)")
print(f"   3. EPSTrendAnalysis (TOP {TOP_N} per window)")
print(f"   4. Summary (ALL tickers combined)")
print(f"   5. InsightsFrequency (ALL tickers)")
print(f"   6. InsightsRankStrength (ALL tickers)")
print(f"   7. InsightsMomentumType (ALL tickers)")
print(f"   8. InsightsPriceVsEstimate (ALL divergences)")
print(f"   9. InsightsCategorySpread (ALL tickers)")
print(f"  10. InsightsYearWise (ALL tickers)")

C:\Users\Sudhir Kulaye\AppData\Local\Temp\ipykernel_11932\1591308200.py:305: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(categorize_momentum)



✅ Excel file created with 10 sheets:
   C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260227_YahooConsensusTrendAnalysis.xlsx

Sheets:
   1. TrendAnalysis (TOP 10 per window)
   2. RevenueTrendAnalysis (TOP 10 per window)
   3. EPSTrendAnalysis (TOP 10 per window)
   4. Summary (ALL tickers combined)
   5. InsightsFrequency (ALL tickers)
   6. InsightsRankStrength (ALL tickers)
   7. InsightsMomentumType (ALL tickers)
   8. InsightsPriceVsEstimate (ALL divergences)
   9. InsightsCategorySpread (ALL tickers)
  10. InsightsYearWise (ALL tickers)
